# 05 - Event Study & Analysis Dataset

Joins storm events, ZHVI panel, and NRI data to produce the final analysis dataset.

**Inputs:**
- `../data/processed/storm_events.pkl` — county-month storm observations
- `../data/processed/nri_panel.pkl` — NRI scores by county-year
- Zillow raw CSV — for pre-storm ZHVI lookups
- Zillow neighbors baseline lookup

**Output:** `../data/processed/analysis_dataset.pkl` — one row per county-month storm event
with complete 12-month post-storm windows

**Measures computed:**
- `auc` — cumulative sum of monthly ZHVI deviation from regional baseline over T+1 to T+12
- `auc_variance` — standard deviation of those 12 monthly deviations
- `pre_trend_annual` — annualized pre-storm drift rate (avg monthly deviation T-6 to T-1) x 12

**Window requirements:**
- 12 months post-storm ZHVI required (incomplete windows dropped)
- 6 months pre-storm ZHVI required for pre_trend_annual
- Latest ZHVI data: February 2026 → latest eligible storm month: February 2025

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

## Load Processed Data

In [2]:
NRI_TYPE = '_smooth'
storms   = pd.read_pickle('../data/processed/storm_events.pkl')
nri      = pd.read_pickle(f'../data/processed/nri_panel{NRI_TYPE}.pkl')

# Load tier-specific ZHVI indexes and baseline lookups
zhvi_idx_mid    = pd.read_pickle('../data/processed/zhvi_idx_mid.pkl')
zhvi_idx_top    = pd.read_pickle('../data/processed/zhvi_idx_top.pkl')
zhvi_idx_bottom = pd.read_pickle('../data/processed/zhvi_idx_bottom.pkl')

baseline_lookup_mid    = pd.read_pickle('../data/processed/baseline_lookup_mid.pkl')
baseline_lookup_top    = pd.read_pickle('../data/processed/baseline_lookup_top.pkl')
baseline_lookup_bottom = pd.read_pickle('../data/processed/baseline_lookup_bottom.pkl')

print(f'Storm events:  {storms.shape}')
print(f'NRI panel:     {nri.shape}')
print(f'ZHVI idx mid:    {len(zhvi_idx_mid):,}')
print(f'ZHVI idx top:    {len(zhvi_idx_top):,}')
print(f'ZHVI idx bottom: {len(zhvi_idx_bottom):,}')

Storm events:  (48978, 9)
NRI panel:     (226368, 14)
ZHVI idx mid:    962,325
ZHVI idx top:    963,270
ZHVI idx bottom: 939,960


## Load Full Zillow ZHVI for Pre-Storm Window

The processed zillow_panel only covers 2020-2025. Pre-storm windows
for early 2020 events require ZHVI from mid-2019, so we load the
full raw dataset and build a lookup.

In [3]:
zhvi_idx_mid    = pd.read_pickle('../data/processed/zhvi_idx_mid.pkl')
zhvi_idx_top    = pd.read_pickle('../data/processed/zhvi_idx_top.pkl')
zhvi_idx_bottom = pd.read_pickle('../data/processed/zhvi_idx_bottom.pkl')

print(f'ZHVI idx mid:    {len(zhvi_idx_mid):,} county-month records')
print(f'ZHVI idx top:    {len(zhvi_idx_top):,} county-month records')
print(f'ZHVI idx bottom: {len(zhvi_idx_bottom):,} county-month records')

ZHVI idx mid:    962,325 county-month records
ZHVI idx top:    963,270 county-month records
ZHVI idx bottom: 939,960 county-month records


## Load Neighbor Baseline Lookup

We need baseline ZHVI for pre-storm months too, not just post-storm.
Recompute baseline from the full zillow panel using the same logic as notebook 04.

In [4]:
PRE_EVENT_MONTHS  = 3
POST_EVENT_MONTHS = 9

baseline_lookup_mid    = pd.read_pickle('../data/processed/baseline_lookup_mid.pkl')
baseline_lookup_top    = pd.read_pickle('../data/processed/baseline_lookup_top.pkl')
baseline_lookup_bottom = pd.read_pickle('../data/processed/baseline_lookup_bottom.pkl')

baseline_idx_mid    = baseline_lookup_mid.set_index(['target_fips', 'storm_year', 'storm_month', 't'])['baseline_zhvi']
baseline_idx_top    = baseline_lookup_top.set_index(['target_fips', 'storm_year', 'storm_month', 't'])['baseline_zhvi']
baseline_idx_bottom = baseline_lookup_bottom.set_index(['target_fips', 'storm_year', 'storm_month', 't'])['baseline_zhvi']

for name, bl in [('mid', baseline_lookup_mid), ('top', baseline_lookup_top), ('bottom', baseline_lookup_bottom)]:
    print(f'{name}: {len(bl):,} rows, {bl.groupby(["target_fips","storm_year","storm_month"]).ngroups:,} events covered')

mid: 241,553 rows, 18,581 events covered
top: 242,814 rows, 18,678 events covered
bottom: 233,727 rows, 17,979 events covered


## Filter to Eligible Storm Events

Drop events without a complete post-storm window.
Latest ZHVI is February 2026, so storm month must be ≤ February 2025.

In [5]:
# Convert year/month to period for cutoff comparison
storms['period'] = pd.to_datetime(
    storms[['year', 'month']].assign(day=1)
)

LATEST_ZHVI = pd.Timestamp('2026-02-28')
CUTOFF      = LATEST_ZHVI - pd.DateOffset(months=POST_EVENT_MONTHS)
eligible    = storms[storms['period'] <= CUTOFF].copy()
n_eligible  = len(eligible)

# --- Isolation filter: drop events with another storm in the PRE/POST window ---
storms_ref = storms[['stcofips', 'year', 'month', 'period']].copy()

neighbors = eligible.merge(
    storms_ref.rename(columns={
        'year': 'other_year',
        'month': 'other_month',
        'period': 'other_period'
    }),
    on='stcofips',
    how='left'
)

neighbors = neighbors[
    ~((neighbors['year'] == neighbors['other_year']) &
      (neighbors['month'] == neighbors['other_month']))
]

neighbors['months_offset'] = (
    (neighbors['other_period'].dt.year  - neighbors['period'].dt.year) * 12 +
    (neighbors['other_period'].dt.month - neighbors['period'].dt.month)
)

contaminated = neighbors[
    (neighbors['months_offset'] >= -PRE_EVENT_MONTHS) &
    (neighbors['months_offset'] <=  POST_EVENT_MONTHS)
][['stcofips', 'year', 'month']].drop_duplicates()

eligible = eligible.merge(
    contaminated.assign(_flag=True),
    on=['stcofips', 'year', 'month'],
    how='left'
)

eligible   = eligible[eligible['_flag'].isna()].drop(columns=['_flag']).copy()
n_isolated = len(eligible)

# --- Damage filter: drop events with no measured damage ---
# isolated = isolated[isolated['total_damage'] > 0].copy()
# n_damage = len(isolated)




print(f'Total storm events:               {len(storms):,}')
print(f'Eligible (complete window):        {n_eligible:,}')
print(f'Isolated (no spillover):           {n_isolated:,}  dropped: {n_eligible - n_isolated:,}  ({(n_eligible - n_isolated)/n_eligible:.1%})')
# print(f'With measured damage:              {n_damage:,}  dropped: {n_isolated - n_damage:,}  ({(n_isolated - n_damage)/n_isolated:.1%})')
print(f'Survival rate (total → final):     {n_isolated/len(storms):.1%}')
print()
print(f"Using {len(eligible)} samples")

Total storm events:               48,978
Eligible (complete window):        44,033
Isolated (no spillover):           2,871  dropped: 41,162  (93.5%)
Survival rate (total → final):     5.9%

Using 2871 samples


## Compute AUC, Variance, and Pre-Trend per Event

In [6]:
from collections import Counter
def compute_event_metrics(row, zhvi_idx, baseline_idx, drop_reasons=Counter()):
    fips        = row['stcofips']
    storm_year  = row['year']
    storm_month = row['month']
    storm_date  = pd.Timestamp(year=storm_year, month=storm_month, day=1)

    # T=0 anchor — county ZHVI only, baseline is already indexed to 100
    county_zhvi_t0   = zhvi_idx.get((fips, storm_year, storm_month))
    baseline_zhvi_t0 = baseline_idx.get((fips, storm_year, storm_month, 0))

    if county_zhvi_t0 is None:
        drop_reasons['county_zhvi_t0_missing'] += 1
        return None
    if baseline_zhvi_t0 is None:
        drop_reasons['baseline_zhvi_t0_missing'] += 1
        return None
    if pd.isna(county_zhvi_t0):
        drop_reasons['county_zhvi_t0_nan'] += 1
        return None
    if pd.isna(baseline_zhvi_t0):
        drop_reasons['baseline_zhvi_t0_nan'] += 1
        return None
    # Assert baseline is indexed to 100 at T=0
    assert abs(baseline_zhvi_t0 - 100) < 0.01, f'Baseline T=0 is {baseline_zhvi_t0}, expected 100'

    # Post-storm window
    post_deviations = []
    for t in range(1, POST_EVENT_MONTHS + 1):
        future = storm_date + pd.DateOffset(months=t)
        county_zhvi   = zhvi_idx.get((fips, future.year, future.month))
        baseline_zhvi = baseline_idx.get((fips, storm_year, storm_month, t))
        if county_zhvi is None:
            drop_reasons[f'post_county_zhvi_missing_t{t}'] += 1
            return None
        if baseline_zhvi is None:
            drop_reasons[f'post_baseline_missing_t{t}'] += 1
            return None
        if pd.isna(county_zhvi) or pd.isna(baseline_zhvi):
            drop_reasons[f'post_nan_t{t}'] += 1
            return None
        # County indexed to 100 at T=0, baseline already indexed
        county_idx = (county_zhvi / county_zhvi_t0) * 100
        post_deviations.append(county_idx - baseline_zhvi)

    # Pre-storm window
    pre_deviations = []
    for t in range(1, PRE_EVENT_MONTHS + 1):
        past = storm_date - pd.DateOffset(months=t)
        county_zhvi   = zhvi_idx.get((fips, past.year, past.month))
        baseline_zhvi = baseline_idx.get((fips, storm_year, storm_month, -t))
        if county_zhvi is None or baseline_zhvi is None or pd.isna(county_zhvi) or pd.isna(baseline_zhvi):
            pre_deviations.append(np.nan)
        else:
            county_idx = (county_zhvi / county_zhvi_t0) * 100
            pre_deviations.append(county_idx - baseline_zhvi)

    post_arr = np.array(post_deviations)
    pre_arr  = np.array(pre_deviations)

    auc               = float(np.sum(post_arr))
    auc_variance      = float(np.std(post_arr, ddof=1))
    pre_trend_monthly = float(np.nanmean(pre_arr)) if not np.all(np.isnan(pre_arr)) else np.nan
    pre_trend_annual  = pre_trend_monthly * 12 if not np.isnan(pre_trend_monthly) else np.nan

    return {
        'auc':              auc,
        'auc_variance':     auc_variance,
        'pre_trend_annual': pre_trend_annual,
        'post_deviations':  post_arr.tolist(),
        'pre_deviations':   pre_arr.tolist(),
    }

print('compute_event_metrics defined')

compute_event_metrics defined


In [7]:
tier_results = []

for tier, zhvi_idx, baseline_idx in [
    ('mid',    zhvi_idx_mid,    baseline_idx_mid),
    ('top',    zhvi_idx_top,    baseline_idx_top),
    ('bottom', zhvi_idx_bottom, baseline_idx_bottom)
]:
    results     = []
    drop_reasons = Counter()
    
    for _, row in eligible.iterrows():
        metrics = compute_event_metrics(row, zhvi_idx, baseline_idx, drop_reasons)
        if metrics is not None:
            results.append({**row.to_dict(), **metrics})
    
    print(f'\n--- {tier} ---')
    print(f'Events with complete windows: {len(results):,}')
    print(f'Drop reasons:')
    for reason, count in sorted(drop_reasons.items(), key=lambda x: -x[1]):
        print(f'  {reason}: {count:,}')
    
    tier_df = pd.DataFrame(results)
    tier_df['tier'] = tier
    tier_results.append(tier_df)

metrics_df = pd.concat(tier_results, ignore_index=True)
print(f'\nTotal rows across all tiers: {len(metrics_df):,}')
print(f'Tiers: {metrics_df["tier"].value_counts().to_dict()}')


--- mid ---
Events with complete windows: 1,751
Drop reasons:
  baseline_zhvi_t0_missing: 886
  county_zhvi_t0_missing: 219
  county_zhvi_t0_nan: 13
  post_nan_t5: 1
  post_nan_t9: 1

--- top ---
Events with complete windows: 1,769
Drop reasons:
  baseline_zhvi_t0_missing: 880
  county_zhvi_t0_missing: 209
  county_zhvi_t0_nan: 10
  post_nan_t5: 1
  post_nan_t9: 1
  post_nan_t4: 1

--- bottom ---
Events with complete windows: 1,659
Drop reasons:
  baseline_zhvi_t0_missing: 899
  county_zhvi_t0_missing: 296
  county_zhvi_t0_nan: 15
  post_nan_t9: 1
  post_nan_t5: 1

Total rows across all tiers: 5,179
Tiers: {'top': 1769, 'mid': 1751, 'bottom': 1659}


## Join Storm and NRI data

In [8]:
nri_cols = ['stcofips', 'storm_year', 'month', 'nri_vintage', 'resl_score', 'resl_value','risk_value', 'eal_valt', 'sovi_score']
metrics_df = metrics_df.merge(
    nri[nri_cols].rename(columns={'storm_year': 'year'}),
    on=['stcofips', 'year', 'month'],
    how='left'
)
print(f'Shape after joins: {metrics_df.shape}')
print(f'Missing NRI: {metrics_df["resl_score"].isnull().sum()}')

Shape after joins: (5179, 22)
Missing NRI: 0


## Validate

In [9]:
missing_nri = metrics_df['resl_score'].isnull()
print(f'Dropping {missing_nri.sum()} rows with missing NRI (CT county restructuring edge case)')
metrics_df = metrics_df[~missing_nri].copy()

assert metrics_df.duplicated(['stcofips', 'year', 'month', 'tier']).sum() == 0, 'Duplicate county-month-tier rows'
assert metrics_df['auc'].notnull().all(), 'Null AUC values'
assert metrics_df['auc_variance'].notnull().all(), 'Null AUC variance values'
assert metrics_df['resl_score'].isnull().sum() == 0, 'Missing NRI scores'

print('All assertions passed')
print(f'\nFinal dataset summary by tier:')
print(metrics_df.groupby('tier')[['auc', 'auc_variance', 'pre_trend_annual', 'log_damage', 'event_count']].describe().round(2))

Dropping 0 rows with missing NRI (CT county restructuring edge case)
All assertions passed

Final dataset summary by tier:
           auc                                                  auc_variance  \
         count  mean    std     min    25%   50%    75%     max        count   
tier                                                                           
bottom  1659.0  1.25  29.56 -167.37 -15.25  0.55  17.59  140.70       1659.0   
mid     1751.0  1.44  23.45 -129.73 -10.94  0.87  13.36  197.85       1751.0   
top     1769.0  1.16  20.42 -107.10  -9.93  1.38  11.75  101.50       1769.0   

              ... log_damage        event_count                             \
        mean  ...        75%    max       count  mean   std  min  25%  50%   
tier          ...                                                            
bottom  1.68  ...       8.52  19.21      1659.0  3.61  4.32  1.0  1.0  2.0   
mid     1.27  ...       8.52  19.21      1751.0  3.61  4.31  1.0  1.0  2.0   
top   

## Export

In [10]:
monthly_rows = []
for _, row in metrics_df.iterrows():
    for t, dev in enumerate(row['post_deviations'], start=1):
        monthly_rows.append({
            'stcofips': row['stcofips'],
            'year':     row['year'],
            'month':    row['month'],
            'tier':     row['tier'],
            'month_t':  t,
            'deviation': dev,
        })

for _, row in metrics_df.iterrows():
    monthly_rows.append({
        'stcofips': row['stcofips'],
        'year':     row['year'],
        'month':    row['month'],
        'tier':     row['tier'],
        'month_t':  0,
        'deviation': 0.0,
    })

monthly_deviations = pd.DataFrame(monthly_rows)

pre_rows = []
for _, row in metrics_df.iterrows():
    for t, dev in enumerate(row['pre_deviations'], start=1):
        if not np.isnan(dev):
            pre_rows.append({
                'stcofips': row['stcofips'],
                'year':     row['year'],
                'month':    row['month'],
                'tier':     row['tier'],
                'month_t':  -t,
                'deviation': dev,
            })

pre_deviations_df = pd.DataFrame(pre_rows)
all_deviations = pd.concat([pre_deviations_df, monthly_deviations], ignore_index=True)
all_deviations.to_csv('../data/processed/monthly_deviations.csv', index=False)
print(f'Exported monthly_deviations.csv: {all_deviations.shape}')
metrics_df = metrics_df.drop(columns=['post_deviations', 'pre_deviations'])
print(f'Dropped cols: post_deviations, pre_deviations')

Exported monthly_deviations.csv: (67315, 6)
Dropped cols: post_deviations, pre_deviations


In [11]:
output_cols = [
    'stcofips', 'year', 'month', 'tier', 'event_type',
    'auc', 'auc_variance', 'pre_trend_annual',
    'event_count', 'total_damage', 'log_damage', 'total_duration_days', 'episode_count',
    'resl_score', 'resl_value', 'risk_value', 'eal_valt', 'sovi_score', 'nri_vintage',
]
out = metrics_df[output_cols].sort_values(['stcofips', 'year', 'month', 'tier']).reset_index(drop=True)

# Filter to events with all 3 tiers present
tier_counts = out.groupby(['stcofips', 'year', 'month'])['tier'].count()
complete    = tier_counts[tier_counts == 3].reset_index()[['stcofips', 'year', 'month']]
out         = out.merge(complete, on=['stcofips', 'year', 'month'], how='inner')

out.to_pickle('../data/processed/analysis_dataset.pkl')
out.to_csv('../data/processed/analysis_dataset.csv', index=False)
print(f'Exported analysis_dataset.pkl and analysis_dataset.csv')
print(f'Shape: {out.shape}')
print(f'Tier counts: {out["tier"].value_counts().to_dict()}')

Exported analysis_dataset.pkl and analysis_dataset.csv
Shape: (4944, 19)
Tier counts: {'bottom': 1648, 'mid': 1648, 'top': 1648}
